# Python for CFD and AI in Fluids  
## A beginner-friendly bridge from programming to fluid-flow computations

This notebook is the **first programming file** for the course **AI in Fluids**.

The goal is not to solve a difficult CFD problem immediately. The goal is to help you understand the programming language and numerical ideas that appear later in CFD and machine learning codes.

Many students can press **Run all**, but that does not mean they understand the code.  
In this notebook, every important Python command is connected to a CFD idea.

| Python idea | Why it matters in CFD | Why it matters in AI |
|---|---|---|
| variables | store physical parameters such as velocity, length, viscosity | store hyperparameters such as learning rate and number of epochs |
| `if` statements | check stability, choose boundary conditions, stop when converged | choose model options, stop training, handle bad data |
| `for` loops | march in time, loop over grid points, compute residuals | loop over epochs or batches |
| arrays | store velocity, pressure, temperature fields | store tensors, images, features, labels |
| functions | organize solvers into reusable pieces | organize models, loss functions, training steps |
| finite differences | approximate derivatives in PDEs | appear in physics-informed losses and residuals |
| residuals | measure whether equations are satisfied | measure loss/error during training |

By the end of this notebook, you should be able to read a simple CFD code and understand:
1. what the variables mean,
2. what arrays represent,
3. why loops are used,
4. how finite differences approximate derivatives,
5. how residuals measure convergence,
6. why this is the same programming style used later in AI models for fluids.

We will use only standard Python, NumPy, and Matplotlib.

## 0. How to use this notebook

Do not only run the cells. For each code block:

1. Read the explanation above the code.
2. Predict what the code will do.
3. Run the code.
4. Change one number.
5. Run it again.
6. Write one sentence explaining what changed.

This is exactly how CFD debugging works. A CFD solver is not a black box.  
A CFD solver is a sequence of small operations: define variables, build a grid, compute derivatives, update fields, apply boundary conditions, and check residuals.

In [ ]:
# Standard packages for this notebook.
# NumPy is the main array/tensor library used in scientific Python.
# Matplotlib is used for plotting fields, profiles, and convergence curves.

import numpy as np
import matplotlib.pyplot as plt

print("NumPy version:", np.__version__)

## 1. Python as a scientific calculator

Before solving a PDE, Python must be able to store and compute physical quantities.

A **variable** is a name attached to a value.  
In CFD, variables often represent physical parameters:

- `rho`: density \(\rho\)
- `U`: characteristic velocity \(U\)
- `L`: characteristic length \(L\)
- `mu`: dynamic viscosity \(\mu\)
- `nu`: kinematic viscosity \(\nu=\mu/\rho\)
- `Re`: Reynolds number \(Re=UL/\nu\)

The Reynolds number compares inertial transport with viscous diffusion.  
It is one of the first dimensionless numbers students should recognize in fluid mechanics.

In [ ]:
# A small fluid-mechanics calculation using Python variables.

rho = 1.0       # density, kg/m^3
U = 1.0         # characteristic velocity, m/s
L = 1.0         # characteristic length, m
nu = 0.01       # kinematic viscosity, m^2/s

Re = U * L / nu

print("Density rho =", rho)
print("Velocity U  =", U)
print("Length L    =", L)
print("Viscosity nu=", nu)
print("Reynolds number Re =", Re)

### Important Python habit: print with context

A beginner often writes:

```python
print(Re)
```

This prints a number, but it does not explain what the number means.  
A better habit is:

```python
print(f"Reynolds number = {Re:.1f}")
```

The `f` before the string means **formatted string**.  
The expression `{Re:.1f}` means: insert the value of `Re` with one digit after the decimal point.

In [ ]:
print(f"Reynolds number = {Re:.1f}")
print(f"Kinematic viscosity = {nu:.4e} m^2/s")

### Try it

Change `nu` from `0.01` to `0.001`.  
What happens to the Reynolds number?  

Expected explanation:

> Smaller viscosity makes Reynolds number larger, so inertia becomes more important relative to viscous diffusion.

## 2. Data types: numbers, strings, and booleans

Python values have types.

Common types in CFD codes:

| Type | Example | CFD meaning |
|---|---|---|
| `int` | `N = 65` | number of grid points |
| `float` | `dt = 0.001` | time step |
| `str` | `"cavity"` | case name |
| `bool` | `plot = True` | turn a feature on/off |

You should always know whether a variable is a number, text, or logical flag.

In [ ]:
N = 65
dt = 0.001
case_name = "lid-driven cavity"
make_plots = True

print(type(N), N)
print(type(dt), dt)
print(type(case_name), case_name)
print(type(make_plots), make_plots)

## 3. Conditions: `if`, `elif`, and `else`

CFD codes constantly make decisions.

Examples:

- If the Reynolds number is small, the flow is likely smooth.
- If the residual is below tolerance, stop iterating.
- If a grid point is on a wall, apply a boundary condition.
- If the time step is too large, warn the user.

The basic structure is:

```python
if condition:
    do something
elif another_condition:
    do something else
else:
    default action
```

Indentation is part of Python syntax. Code inside an `if` block must be indented.

In [ ]:
Re = 100

if Re < 1:
    regime = "creeping or Stokes-like flow"
elif Re < 1000:
    regime = "laminar flow is expected for many simple internal benchmarks"
else:
    regime = "inertia is strong; instabilities or turbulence may appear depending on geometry"

print(f"Re = {Re}")
print("Estimated regime:", regime)

### Conditions for numerical safety

A very common CFD pattern is to check whether a numerical parameter is safe.  
For an explicit diffusion update, a simplified stability condition is

\[
\Delta t \le C \frac{\Delta x^2}{\nu}.
\]

The exact constant depends on the equation and discretization.  
The important programming idea is that a CFD code should **check** stability instead of silently producing nonsense.

In [ ]:
dx = 1.0 / (N - 1)
nu = 0.01
dt = 0.001

stability_limit = 0.5 * dx**2 / nu

print(f"dx = {dx:.5f}")
print(f"dt = {dt:.5e}")
print(f"estimated stability limit = {stability_limit:.5e}")

if dt <= stability_limit:
    print("Time step looks safe for this simple diffusion estimate.")
else:
    print("WARNING: time step may be too large.")

### Exercise: write your own condition

Write an `if` statement that prints:

- `"coarse grid"` if `N < 50`
- `"medium grid"` if `50 <= N < 150`
- `"fine grid"` if `N >= 150`

This is not just a Python exercise. CFD reports often compare coarse, medium, and fine grids.

In [ ]:
# Student exercise:
N = 65

# TODO: complete this block
if N < 50:
    grid_type = "coarse grid"
elif N < 150:
    grid_type = "medium grid"
else:
    grid_type = "fine grid"

print(f"N = {N}: {grid_type}")

## 4. Lists, tuples, and dictionaries

CFD simulations usually store many settings.

A **list** is an ordered collection that can be changed.  
A **tuple** is an ordered collection usually treated as fixed.  
A **dictionary** maps names to values.

Dictionaries are very useful for simulation parameters because they make code readable.

In [ ]:
# A list of grid sizes
grid_sizes = [33, 65, 129]

# A tuple for a 2D velocity vector
lid_velocity = (1.0, 0.0)

# A dictionary for a CFD case
case = {
    "name": "Ghia cavity warmup",
    "Re": 100,
    "N": 65,
    "dt": 0.001,
    "steps": 15000,
    "method": "finite difference"
}

print("Grid sizes:", grid_sizes)
print("Lid velocity:", lid_velocity)
print("Case dictionary:", case)
print("The Reynolds number is", case["Re"])

### Why dictionaries matter

Compare these two lines:

```python
run(100, 65, 0.001, 15000)
```

and

```python
run(Re=100, N=65, dt=0.001, steps=15000)
```

The second one is easier to read. In CFD and machine learning, readability prevents mistakes.

## 5. Loops: repeating an operation

A CFD solver repeats operations many times.

Examples:

- repeat time steps,
- repeat Jacobi iterations for a Poisson equation,
- loop over grid points,
- loop over epochs in machine learning.

The simplest loop is a `for` loop.

In [ ]:
# A simple for loop.
for step in range(5):
    print("step =", step)

`range(5)` produces the numbers `0, 1, 2, 3, 4`.

Python indexing starts from zero.  
This is important because arrays also start from index zero.

In [ ]:
# A loop that mimics residual decay.
residual = 1.0

for iteration in range(10):
    residual = 0.6 * residual
    print(f"iteration {iteration:02d}: residual = {residual:.4e}")

### `while` loops and convergence

A `while` loop repeats until a condition becomes false.

This is closer to how many CFD solvers work:

```python
while residual > tolerance:
    update_solution()
    compute_residual()
```

A solver should not stop just because it has completed a fixed number of steps.  
It should ideally stop because the residual is small enough.

In [ ]:
residual = 1.0
tolerance = 1e-3
iteration = 0
max_iterations = 100

while residual > tolerance and iteration < max_iterations:
    residual *= 0.7
    iteration += 1

print("Stopped at iteration:", iteration)
print(f"Final residual = {residual:.3e}")

if residual <= tolerance:
    print("Converged.")
else:
    print("Stopped because maximum iterations was reached.")

## 6. Functions: reusable blocks of code

A function takes inputs and returns outputs.  
Functions make CFD codes easier to read and test.

Instead of writing the Reynolds number formula everywhere, define it once.

In [ ]:
def reynolds_number(U, L, nu):
    # Compute Reynolds number Re = U L / nu.
    return U * L / nu

print(reynolds_number(U=1.0, L=1.0, nu=0.01))

### Functions should be small and testable

A good CFD code is not one giant block.  
A good CFD code contains small functions such as:

- `build_grid`
- `apply_boundary_conditions`
- `compute_laplacian`
- `compute_residual`
- `plot_centerline_velocity`

Small functions are easier to debug and easier to reuse in AI workflows.

In [ ]:
def classify_reynolds(Re):
    # Return a short qualitative description of the flow regime.
    if Re < 1:
        return "creeping-flow regime"
    elif Re < 1000:
        return "laminar benchmark regime"
    else:
        return "high-Re regime; instabilities may matter"

for Re_test in [0.1, 100, 5000]:
    print(f"Re = {Re_test:7.1f}: {classify_reynolds(Re_test)}")

## 7. NumPy arrays: fields on a grid

In CFD, a field is a quantity defined at many points:

- \(u(x,y)\): horizontal velocity
- \(v(x,y)\): vertical velocity
- \(p(x,y)\): pressure
- \(T(x,y)\): temperature
- \(\omega(x,y)\): vorticity

On a computer, a field becomes an array.

For a two-dimensional grid, we usually store data in an array with shape:

```python
(Ny, Nx)
```

The first index is the row or \(y\)-direction.  
The second index is the column or \(x\)-direction.

So `u[j, i]` means the value at \(y_j, x_i\).

In [ ]:
Nx = 65
Ny = 65

x = np.linspace(0.0, 1.0, Nx)
y = np.linspace(0.0, 1.0, Ny)

X, Y = np.meshgrid(x, y)

print("x shape:", x.shape)
print("y shape:", y.shape)
print("X shape:", X.shape)
print("Y shape:", Y.shape)

In [ ]:
# Create a smooth scalar field.
phi = np.sin(np.pi * X) * np.sin(np.pi * Y)

plt.figure(figsize=(5, 4))
plt.contourf(X, Y, phi, levels=30)
plt.colorbar(label="phi")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Example scalar field on a 2D grid")
plt.tight_layout()
plt.show()

### Why this matters for machine learning

A 2D CFD field is like an image:

- an image has pixels,
- a CFD field has grid values.

A CNN, U-Net, or neural operator sees a CFD field as a tensor.  
This is why learning arrays carefully is essential before using AI in fluids.

## 8. Indexing and slicing

Slicing means selecting part of an array.

Examples:

- `u[0, :]` = bottom row
- `u[-1, :]` = top row
- `u[:, 0]` = left column
- `u[:, -1]` = right column
- `u[1:-1, 1:-1]` = interior points

Boundary conditions in CFD are often implemented with slicing.

In [ ]:
u = np.zeros((Ny, Nx))

# Lid-driven cavity boundary condition:
# top wall moves with u = 1, all other walls have u = 0.
u[-1, :] = 1.0

print("Bottom wall average u:", np.mean(u[0, :]))
print("Top wall average u   :", np.mean(u[-1, :]))
print("Interior average u   :", np.mean(u[1:-1, 1:-1]))

plt.figure(figsize=(5, 4))
plt.imshow(u, origin="lower", extent=[0, 1, 0, 1])
plt.colorbar(label="u")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Top-wall boundary condition")
plt.tight_layout()
plt.show()

## 9. Finite differences: turning derivatives into array operations

The derivative of a function measures how rapidly it changes.  
CFD equations contain derivatives such as:

\[
\frac{\partial u}{\partial x}, \qquad
\frac{\partial^2 u}{\partial x^2}, \qquad
\nabla^2 u.
\]

On a grid, derivatives are approximated by finite differences.

For a 1D array \(f_i=f(x_i)\), a centered first derivative is

\[
\left.\frac{df}{dx}\right|_i
\approx
\frac{f_{i+1}-f_{i-1}}{2\Delta x}.
\]

A centered second derivative is

\[
\left.\frac{d^2f}{dx^2}\right|_i
\approx
\frac{f_{i+1}-2f_i+f_{i-1}}{\Delta x^2}.
\]

These formulas are the computational building blocks of CFD.

In [ ]:
# Test finite differences on a known function.
# f(x) = sin(pi x)
# df/dx = pi cos(pi x)

N = 101
x = np.linspace(0.0, 1.0, N)
dx = x[1] - x[0]

f = np.sin(np.pi * x)
df_exact = np.pi * np.cos(np.pi * x)

df_num = np.zeros_like(f)
df_num[1:-1] = (f[2:] - f[:-2]) / (2.0 * dx)

error = np.linalg.norm(df_num[1:-1] - df_exact[1:-1]) / np.linalg.norm(df_exact[1:-1])

print(f"Relative derivative error = {error:.3e}")

plt.figure(figsize=(6, 4))
plt.plot(x, df_exact, label="exact")
plt.plot(x, df_num, "--", label="finite difference")
plt.xlabel("x")
plt.ylabel("df/dx")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

### The 2D Laplacian stencil

For a field \(\phi(x,y)\), the Laplacian is

\[
\nabla^2 \phi =
\frac{\partial^2 \phi}{\partial x^2}
+
\frac{\partial^2 \phi}{\partial y^2}.
\]

On a square grid with spacing \(\Delta x=\Delta y\), a common finite-difference stencil is:

\[
\nabla^2\phi_{j,i}
\approx
\frac{
\phi_{j,i+1} + \phi_{j,i-1} + \phi_{j+1,i} + \phi_{j-1,i} - 4\phi_{j,i}
}{\Delta x^2}.
\]

This is a local operation: each point talks to its four nearest neighbors.  
This locality is also why convolutional neural networks are natural for grid-based CFD fields.

In [ ]:
def laplacian_2d(phi, dx):
    # Compute the 2D Laplacian using a five-point finite-difference stencil.
    # The boundary values are left as zero in the returned array.
    lap = np.zeros_like(phi)
    lap[1:-1, 1:-1] = (
        phi[1:-1, 2:]     # right neighbor
        + phi[1:-1, :-2]  # left neighbor
        + phi[2:, 1:-1]   # top neighbor
        + phi[:-2, 1:-1]  # bottom neighbor
        - 4.0 * phi[1:-1, 1:-1]
    ) / dx**2
    return lap

N = 65
x = np.linspace(0, 1, N)
y = np.linspace(0, 1, N)
X, Y = np.meshgrid(x, y)
dx = x[1] - x[0]

phi = np.sin(np.pi * X) * np.sin(np.pi * Y)
lap_phi = laplacian_2d(phi, dx)

plt.figure(figsize=(5, 4))
plt.contourf(X, Y, lap_phi, levels=30)
plt.colorbar(label="Laplacian")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Numerical Laplacian of a smooth field")
plt.tight_layout()
plt.show()

## 10. Boundary conditions: values imposed on the edges

A PDE does not have a unique solution unless boundary conditions are specified.

Common types:

| Boundary condition | Meaning | Example |
|---|---|---|
| Dirichlet | value is specified | \(u=0\) at a no-slip wall |
| Neumann | derivative is specified | \(\partial T/\partial n=0\) for adiabatic wall |
| Periodic | left and right sides connect | turbulence in a periodic box |

The lid-driven cavity has no-slip walls:
- bottom, left, right walls: \(u=v=0\)
- top lid: \(u=U,\;v=0\)

The following function applies a simple lid-driven-cavity velocity boundary condition.

In [ ]:
def apply_cavity_velocity_bc(u, v, lid_speed=1.0):
    # Apply velocity boundary conditions for a lid-driven cavity.

    # Bottom wall
    u[0, :] = 0.0
    v[0, :] = 0.0

    # Top moving lid
    u[-1, :] = lid_speed
    v[-1, :] = 0.0

    # Left wall
    u[:, 0] = 0.0
    v[:, 0] = 0.0

    # Right wall
    u[:, -1] = 0.0
    v[:, -1] = 0.0

    return u, v

u = np.zeros((65, 65))
v = np.zeros((65, 65))
u, v = apply_cavity_velocity_bc(u, v, lid_speed=1.0)

print("top-wall u average:", np.mean(u[-1, :]))
print("left-wall u average:", np.mean(u[:, 0]))

## 11. Vectorization: replacing slow loops with array operations

A beginner might compute an update with nested loops:

```python
for j in range(1, Ny-1):
    for i in range(1, Nx-1):
        new[j,i] = ...
```

This is easy to understand, but it is slow for large grids.

NumPy allows vectorized operations over entire slices:

```python
new[1:-1,1:-1] = ...
```

Vectorization is important because:
1. it is faster,
2. it makes the mathematical stencil clearer,
3. it resembles tensor operations in machine learning.

In [ ]:
# Compare loop and vectorized versions of a Laplacian update.

N = 100
phi = np.random.rand(N, N)
dx = 1.0 / (N - 1)

# Loop version
lap_loop = np.zeros_like(phi)
for j in range(1, N - 1):
    for i in range(1, N - 1):
        lap_loop[j, i] = (
            phi[j, i+1] + phi[j, i-1] + phi[j+1, i] + phi[j-1, i]
            - 4.0 * phi[j, i]
        ) / dx**2

# Vectorized version
lap_vec = np.zeros_like(phi)
lap_vec[1:-1, 1:-1] = (
    phi[1:-1, 2:] + phi[1:-1, :-2] + phi[2:, 1:-1] + phi[:-2, 1:-1]
    - 4.0 * phi[1:-1, 1:-1]
) / dx**2

difference = np.max(np.abs(lap_loop - lap_vec))
print(f"Maximum difference between loop and vectorized result = {difference:.3e}")

## 12. A simple iterative solver: Jacobi solution of Poisson's equation

Many incompressible-flow algorithms need to solve a Poisson equation:

\[
\nabla^2 p = b.
\]

For teaching, we first solve a simpler manufactured problem:

\[
\nabla^2 \phi = b,
\]

with zero boundary values. We use a **Jacobi iteration**, which repeatedly updates each interior point from its neighbors.

This is not the fastest solver, but it is excellent for learning:
- arrays,
- loops,
- residuals,
- convergence,
- boundary conditions.

In [ ]:
def poisson_jacobi(b, dx, n_iters=500, report_every=100):
    # Solve Laplacian(phi) = b on a square domain with phi=0 boundaries.
    phi = np.zeros_like(b)
    residual_history = []

    for it in range(1, n_iters + 1):
        old = phi.copy()

        phi[1:-1, 1:-1] = 0.25 * (
            old[1:-1, 2:] + old[1:-1, :-2]
            + old[2:, 1:-1] + old[:-2, 1:-1]
            - dx**2 * b[1:-1, 1:-1]
        )

        # Compute residual r = Laplacian(phi) - b
        r = laplacian_2d(phi, dx) - b
        res = np.linalg.norm(r[1:-1, 1:-1]) / (np.linalg.norm(b[1:-1, 1:-1]) + 1e-14)
        residual_history.append(res)

        if it % report_every == 0:
            print(f"iteration {it:5d}: residual = {res:.3e}")

    return phi, np.array(residual_history)

N = 65
x = np.linspace(0, 1, N)
y = np.linspace(0, 1, N)
X, Y = np.meshgrid(x, y)
dx = x[1] - x[0]

# Manufactured source term
b = -2.0 * np.pi**2 * np.sin(np.pi * X) * np.sin(np.pi * Y)

phi, residuals = poisson_jacobi(b, dx, n_iters=600, report_every=200)

plt.figure(figsize=(5, 4))
plt.semilogy(residuals)
plt.xlabel("Jacobi iteration")
plt.ylabel("relative residual")
plt.grid(alpha=0.3)
plt.title("Poisson residual history")
plt.tight_layout()
plt.show()

### What did we just learn?

This Poisson example is small, but it contains the logic of many CFD solvers:

1. create a grid,
2. store unknowns in arrays,
3. update interior points,
4. apply boundary conditions,
5. compute a residual,
6. stop when the residual is small.

Later, when we solve incompressible Navier--Stokes, the pressure or streamfunction equation plays a similar role.

## 13. Time marching: a simple 1D diffusion equation

Consider the diffusion equation:

\[
\frac{\partial u}{\partial t}
=
\nu \frac{\partial^2 u}{\partial x^2}.
\]

This equation describes smoothing. A sharp initial profile becomes smoother over time.

This is a useful warmup because:
- it contains a time loop,
- it uses finite differences,
- it has a stability restriction,
- it illustrates the role of viscosity.

In [ ]:
# 1D diffusion example

N = 101
x = np.linspace(0, 1, N)
dx = x[1] - x[0]
nu = 0.01
dt = 0.2 * dx**2 / nu  # safe explicit time step
steps = 200

u = np.zeros(N)
u[(x > 0.4) & (x < 0.6)] = 1.0  # square pulse

history = [u.copy()]

for step in range(steps):
    old = u.copy()
    u[1:-1] = old[1:-1] + nu * dt / dx**2 * (old[2:] - 2*old[1:-1] + old[:-2])

    # Dirichlet boundaries
    u[0] = 0.0
    u[-1] = 0.0

    if step in [10, 50, 100, 199]:
        history.append(u.copy())

plt.figure(figsize=(6, 4))
for k, profile in enumerate(history):
    plt.plot(x, profile, label=f"snapshot {k}")
plt.xlabel("x")
plt.ylabel("u")
plt.title("1D diffusion smooths a sharp profile")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 14. A tiny machine-learning-style example using only NumPy

Before TensorFlow, it is useful to understand the basic structure of learning:

1. choose a model,
2. make predictions,
3. compute a loss,
4. update parameters,
5. repeat.

This is similar to CFD iteration:

| CFD | Machine learning |
|---|---|
| unknown field | model parameters |
| PDE residual | loss |
| iteration/time step | epoch |
| convergence history | training-loss history |

Here we fit a line \(y=ax+b\) using gradient descent.

In [ ]:
# Generate synthetic data
rng = np.random.default_rng(2)
x_data = np.linspace(0, 1, 50)
y_true = 2.0 * x_data + 0.5
y_data = y_true + 0.05 * rng.normal(size=x_data.shape)

# Initial model parameters
a = 0.0
b = 0.0
learning_rate = 0.5
loss_history = []

for epoch in range(200):
    y_pred = a * x_data + b
    error = y_pred - y_data
    loss = np.mean(error**2)
    loss_history.append(loss)

    # Gradients of mean squared error
    grad_a = 2.0 * np.mean(error * x_data)
    grad_b = 2.0 * np.mean(error)

    # Parameter update
    a -= learning_rate * grad_a
    b -= learning_rate * grad_b

print(f"Learned a = {a:.3f}")
print(f"Learned b = {b:.3f}")

plt.figure(figsize=(6, 4))
plt.scatter(x_data, y_data, label="data")
plt.plot(x_data, a*x_data + b, label="learned model")
plt.plot(x_data, y_true, "--", label="true line")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 4))
plt.semilogy(loss_history)
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("Training loss history")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 15. How this prepares you for the lid-driven cavity CFD code

The Ghia cavity solver will look more complicated, but it uses the same ideas:

| This notebook | Cavity code |
|---|---|
| `N`, `dx`, `dt`, `Re` | grid spacing, time step, Reynolds number |
| 2D arrays | streamfunction, vorticity, velocity |
| finite differences | derivatives in Navier--Stokes |
| boundary slices | no-slip wall conditions |
| loops | time marching and Poisson iterations |
| residual history | convergence to steady state |
| validation plot | comparison with Ghia et al. |

The student should now be able to open the cavity solver and identify:
- where the grid is created,
- where the boundary conditions are applied,
- where derivatives are computed,
- where iterations happen,
- where errors against benchmark data are calculated.

## 16. Required student deliverables for this Python warmup

Submit a short PDF or notebook export containing:

1. Your Reynolds-number calculation and one sentence explaining what happens when viscosity decreases.
2. Your `if/elif/else` grid classifier.
3. A plot of the numerical derivative of \(\sin(\pi x)\) compared with the exact derivative.
4. A plot of the Poisson residual history.
5. A plot of the 1D diffusion equation showing smoothing over time.
6. A short paragraph connecting residuals in CFD to losses in machine learning.

This is a programming assignment, not just a clicking assignment.  
Your code must run, but your explanations are equally important.

## Article-output contract

<!-- MIE690A article-aligned validation v3 -->

**Role:** Foundational or supporting notebook; see ARTICLE_FIGURE_MAP.md for its evidence dependency.

All manuscript-facing figures must be generated from retained numerical/model outputs through the documented notebook or shared helper, saved under `results/`, and accompanied by machine-readable metrics. Do not redraw curves by eye or substitute a screenshot for a solver-to-reference comparison. The complete ownership table and exact output filenames are in [`ARTICLE_FIGURE_MAP.md`](../../ARTICLE_FIGURE_MAP.md).
